# Checking which Tables Exist

In [1]:
# Verification: list all tables in the database
import sqlite3
from pathlib import Path

db_path = Path.cwd().parent / 'boilest.db'
conn = sqlite3.connect(db_path)
cur = conn.cursor()
cur.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cur.fetchall()
print('Tables in boilest.db:', tables)
conn.close()


Tables in boilest.db: [('sqlite_sequence',), ('directories',), ('error',), ('queue',), ('completed',)]


# Setting up the Directories Table

In [ ]:
import sqlite3
import uuid
from pathlib import Path

db_path = Path.cwd().parent / 'boilest.db'
conn = sqlite3.connect(db_path)
cur = conn.cursor()

# Check if the 'directories' table exists
cur.execute(
    "SELECT name FROM sqlite_master WHERE type='table' AND name='directories';"
)
exists = cur.fetchone()
if exists:
    print("Table 'directories' already exists.")
else:
    cur.execute("""CREATE TABLE directories (
        guid TEXT PRIMARY KEY,
        path TEXT NOT NULL UNIQUE,
        added_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP
    );""")
    conn.commit()
    print("Table 'directories' created.")

conn.close()


# Adding rows to directories

In [2]:
import sqlite3
import uuid
from pathlib import Path

def add_directory_path(path):
    """Add a directory path to the directories table"""
    db_path = Path.cwd().parent / 'boilest.db'
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    try:
        guid = str(uuid.uuid4())
        cur.execute('INSERT INTO directories (guid, path) VALUES (?, ?)', (guid, path))
        conn.commit()
        print(f'Inserted: {path} with guid: {guid}')
    except sqlite3.IntegrityError:
        print(f'Path already exists: {path}')
    finally:
        conn.close()

# Add /media directory
add_directory_path('/Boil/Media/TV')
add_directory_path('/Boil/Media/Movies')
add_directory_path('/Boil/Media/Anime')

Path already exists: /Boil/Media/TV
Path already exists: /Boil/Media/Movies
Path already exists: /Boil/Media/Anime


# Creating the Queue Table

In [13]:
import sqlite3
import uuid
from pathlib import Path

def create_queue_table():
    db_path = 'boilest.db'
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    
    # Check if the 'queue' table exists
    cur.execute(
        "SELECT name FROM sqlite_master WHERE type='table' AND name='queue';"
    )
    exists = cur.fetchone()
    
    if exists:
        print("Table 'queue' already exists.")
    else:
        cur.execute("""CREATE TABLE queue (
            directory_guid TEXT NOT NULL,
            file_guid TEXT PRIMARY KEY NOT NULL,
            directory_path TEXT NOT NULL,
            input_file_name TEXT NOT NULL,
            output_file_name TEXT NOT NULL,
            before_file_size INTEGER NOT NULL,
            after_file_size INTEGER,
            ffmpeg_string TEXT NOT NULL,
            datetime_added TEXT NOT NULL,
            datetime_pulled TEXT,
            datetime_encoded TEXT
        );""")
        conn.commit()
        print("Table 'queue' created.")
    
    conn.close()

# Create the queue table
create_queue_table()

Table 'queue' created.


# Adding the Error Table   

In [7]:
import sqlite3
import uuid
from pathlib import Path

def create_error_table():
    db_path = Path.cwd().parent / 'boilest.db'
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    
    # Check if the 'error' table exists
    cur.execute(
        "SELECT name FROM sqlite_master WHERE type='table' AND name='error';"
    )
    exists = cur.fetchone()
    
    if exists:
        print("Table 'error' already exists.")
    else:
        cur.execute("""CREATE TABLE error (
            guid TEXT PRIMARY KEY,
            queued_file_guid TEXT NOT NULL,        
            reason TEXT NOT NULL,
            additional_detail TEXT,
            date_added TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP
        );""")
        conn.commit()
        print("Table 'error' created.")
    
    conn.close()

# Create the error table
create_error_table()


Table 'error' already exists.


# Adding the ffmpeg settings table

# Delete All Records

In [9]:
import sqlite3
from pathlib import Path

def delete_all_records_except_directories():
    """
    Delete all records from all tables in boilest.db except the directories table.
    This keeps directory paths intact while clearing files, encoding history, and errors.
    """
    db_path = Path.cwd().parent / 'boilest.db'
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    
    try:
        # Get all table names
        cur.execute("SELECT name FROM sqlite_master WHERE type='table';")
        tables = [row[0] for row in cur.fetchall()]
        
        print(f"Found tables: {tables}")
        print("-" * 80)
        
        # Tables to skip (don't delete from these)
        skip_tables = {'directories', 'sqlite_sequence'}
        
        deleted_count = 0
        
        for table in tables:
            if table in skip_tables:
                print(f"⊘ Skipping table: {table}")
                continue
            
            # Delete all records from this table
            cur.execute(f"DELETE FROM {table}")
            rows_affected = cur.rowcount
            
            if rows_affected > 0:
                print(f"✓ Deleted {rows_affected} records from table: {table}")
                deleted_count += rows_affected
            else:
                print(f"⊘ No records to delete from table: {table}")
        
        conn.commit()
        
        print("-" * 80)
        print(f"✓ Total {deleted_count} records deleted")
        print("✓ Directories table preserved with all paths intact")
        
        return {
            'success': True,
            'tables_cleaned': [t for t in tables if t not in skip_tables],
            'total_records_deleted': deleted_count
        }
        
    except Exception as e:
        conn.rollback()
        print(f"✗ Error deleting records: {e}")
        return {'success': False, 'error': str(e)}
    
    finally:
        conn.close()

# Execute the delete function
print("Deleting all records except from directories table...")
print("=" * 80)
result = delete_all_records_except_directories()
print(f"\nResult: {result}")

Deleting all records except from directories table...
Found tables: ['sqlite_sequence', 'directories', 'error', 'completed', 'queue']
--------------------------------------------------------------------------------
⊘ Skipping table: sqlite_sequence
⊘ Skipping table: directories
⊘ No records to delete from table: error
⊘ No records to delete from table: completed
⊘ No records to delete from table: queue
--------------------------------------------------------------------------------
✓ Total 0 records deleted
✓ Directories table preserved with all paths intact

Result: {'success': True, 'tables_cleaned': ['error', 'completed', 'queue'], 'total_records_deleted': 0}
